# Lesson 13: Build a tiny decoder language model

Lesson 12 produced token-plus-position vectors. Here we pass them through two pre-LayerNorm Transformer blocks, project the final vectors to one score per vocabulary token, measure next-token cross-entropy, and generate a short continuation.

The model is intentionally tiny and uses fixed deterministic weights rather than training. This keeps the forward pass readable and makes its values comparable with the C and Java versions. The four steps group the ideas for this notebook; the other languages use the same numbered steps across methods in [C](../c/src/decoder_model_main.c) and [Java](../java/src/main/java/learning/karpathy/gpt1/DecoderLanguageModelLesson.java). See the [shared lesson map](../README.md#run-lessons-by-step).

References: [PyTorch Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [LayerNorm](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html), [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html), and [cross-entropy](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html).


## Shared setup: a fixed, readable model

The small vocabulary is `a` through `e`; the two short sequences provide batch and next-token targets. Each helper initializes weights from an integer formula, so every language starts from the same values.


In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F

BATCH_SIZE = 2
BLOCK_SIZE = 4
VOCAB_SIZE = 5
CHANNELS = 4
HEADS = 2
HEAD_SIZE = CHANNELS // HEADS
FEED_FORWARD_SIZE = 4 * CHANNELS
LAYERS = 2
GENERATED_TOKENS = 5
VOCABULARY = ["a", "b", "c", "d", "e"]
TOKEN_IDS = torch.tensor([[0, 1, 0, 2], [1, 2, 3, 0]], dtype=torch.long)
TARGET_IDS = torch.tensor([[1, 0, 2, 3], [2, 3, 0, 1]], dtype=torch.long)
SAMPLE_THRESHOLDS = [0.19, 0.47, 0.81, 0.33, 0.68]


def deterministic_weight(seed, output_channel, input_channel, modulus, scale):
    pattern = (seed * 5 + (output_channel + 1) * (input_channel + 2)) % modulus
    return (pattern - modulus // 2) * scale


def deterministic_bias(seed, channel):
    return ((seed + channel * 2) % 5 - 2) * 0.01


def initialize_linear(linear, weight_seed, bias_seed, weight_modulus, weight_scale):
    with torch.no_grad():
        rows, columns = linear.weight.shape
        weights = [
            [deterministic_weight(weight_seed, row, column, weight_modulus, weight_scale)
             for column in range(columns)]
            for row in range(rows)
        ]
        linear.weight.copy_(torch.tensor(weights, dtype=linear.weight.dtype))
        if linear.bias is not None:
            biases = [deterministic_bias(bias_seed, channel) for channel in range(rows)]
            linear.bias.copy_(torch.tensor(biases, dtype=linear.bias.dtype))




### Model components: causal attention

Each head learns separate query, key, and value projections. A lower-triangular mask prevents a position from reading future tokens; scaled scores become weights with softmax. The multi-head module runs two such heads, joins their channels, and projects back to the model width.


In [ ]:
class CausalAttentionHead(nn.Module):
    def __init__(self, layer_index, head_index):
        super().__init__()
        self.query = nn.Linear(CHANNELS, HEAD_SIZE, bias=False)
        self.key = nn.Linear(CHANNELS, HEAD_SIZE, bias=False)
        self.value = nn.Linear(CHANNELS, HEAD_SIZE, bias=False)
        self.layer_index = layer_index
        self.head_index = head_index

    def initialize_deterministic(self):
        seed = self.layer_index * 10 + self.head_index * 3
        for projection, offset in ((self.query, 1), (self.key, 2), (self.value, 3)):
            initialize_linear(projection, seed + offset, 0, 11, 0.08)

    def forward(self, x):
        _, time_steps, _ = x.shape
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)
        scores = query @ key.transpose(-2, -1) / math.sqrt(HEAD_SIZE)
        future_positions = torch.triu(
            torch.ones(time_steps, time_steps, dtype=torch.bool, device=x.device), diagonal=1
        )
        scores = scores.masked_fill(future_positions, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return weights @ value



class MultiHeadAttention(nn.Module):
    def __init__(self, layer_index):
        super().__init__()
        self.heads = nn.ModuleList(
            CausalAttentionHead(layer_index, head_index) for head_index in range(HEADS)
        )
        self.projection = nn.Linear(CHANNELS, CHANNELS)
        self.layer_index = layer_index

    def initialize_deterministic(self):
        for head in self.heads:
            head.initialize_deterministic()
        initialize_linear(self.projection, self.layer_index * 10 + 7,
                          self.layer_index * 10 + 8, 11, 0.05)

    def forward(self, x):
        joined_heads = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.projection(joined_heads)




### Model components: feed-forward network and Transformer block

The feed-forward network expands each token vector from four channels to sixteen, applies ReLU, then projects back. It works independently at each time position. The block applies LayerNorm before attention and before the feed-forward network, adding each branch through a residual connection.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, layer_index):
        super().__init__()
        self.first = nn.Linear(CHANNELS, FEED_FORWARD_SIZE)
        self.second = nn.Linear(FEED_FORWARD_SIZE, CHANNELS)
        self.dropout = nn.Dropout(0.1)
        self.layer_index = layer_index

    def initialize_deterministic(self):
        seed = self.layer_index * 13
        initialize_linear(self.first, seed + 9, seed + 10, 11, 0.04)
        initialize_linear(self.second, seed + 11, seed + 12, 11, 0.03)

    def forward(self, x):
        hidden = F.relu(self.first(x))
        return self.dropout(self.second(hidden))



class TransformerBlock(nn.Module):
    def __init__(self, layer_index):
        super().__init__()
        self.first_norm = nn.LayerNorm(CHANNELS, eps=1e-5)
        self.attention = MultiHeadAttention(layer_index)
        self.second_norm = nn.LayerNorm(CHANNELS, eps=1e-5)
        self.feed_forward = FeedForward(layer_index)

    def initialize_deterministic(self):
        self.attention.initialize_deterministic()
        self.feed_forward.initialize_deterministic()

    def forward(self, x):
        x = x + self.attention(self.first_norm(x))
        x = x + self.feed_forward(self.second_norm(x))
        return x




### Compose the decoder language model

`TinyGPT` combines token and position embeddings, two blocks, a final LayerNorm, and a linear language-model head. Its forward method returns intermediate tensors so the next cells can inspect each stage.


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, CHANNELS)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, CHANNELS)
        self.blocks = nn.ModuleList(TransformerBlock(layer) for layer in range(LAYERS))
        self.final_norm = nn.LayerNorm(CHANNELS, eps=1e-5)
        self.language_model_head = nn.Linear(CHANNELS, VOCAB_SIZE)

    def initialize_deterministic(self):
        with torch.no_grad():
            token_table = [
                [((token * 5 + channel * 3 + 1) % 11 - 5) * 0.1
                 for channel in range(CHANNELS)]
                for token in range(VOCAB_SIZE)
            ]
            position_table = [
                [((position * 7 + channel * 2 + 2) % 9 - 4) * 0.1
                 for channel in range(CHANNELS)]
                for position in range(BLOCK_SIZE)
            ]
            self.token_embedding.weight.copy_(torch.tensor(token_table, dtype=torch.float64))
            self.position_embedding.weight.copy_(torch.tensor(position_table, dtype=torch.float64))
        for block in self.blocks:
            block.initialize_deterministic()
        initialize_linear(self.language_model_head, 41, 42, 11, 0.05)

    def forward(self, token_ids):
        _, time_steps = token_ids.shape
        positions = torch.arange(time_steps, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        embedding_output = x
        block_outputs = []
        for block in self.blocks:
            x = block(x)
            block_outputs.append(x)
        logits = self.language_model_head(self.final_norm(x))
        return embedding_output, block_outputs, logits


model = TinyGPT().to(dtype=torch.float64)
model.initialize_deterministic()
model.eval()  # disables dropout; fixed weights are not being trained in this lesson


## Step 1 — embed tokens and add positions

Each integer selects a row from the token table. The position table gives the vector's place in the sequence; adding the two gives the input tensor for the first Transformer block. Shapes are `[batch, time, channels]`.


In [ ]:
with torch.inference_mode():
    embedding_output, block_outputs, logits = model(TOKEN_IDS)

print("Vocabulary IDs: a=0, b=1, c=2, d=3, e=4")
print(f"Token IDs, shape [batch, time] = {list(TOKEN_IDS.shape)}:")
print(TOKEN_IDS)
print(f"Input to the Transformer blocks, shape [batch, time, channels] = {list(embedding_output.shape)}:")
print(embedding_output[0])


## Step 2 — pass through two Transformer blocks

Each block applies LayerNorm before causal multi-head attention, adds the attention result back to the residual stream, then repeats the pattern with a position-wise feed-forward network. Only the tiny head and context sizes are simplified; the block structure matches a decoder block.


In [ ]:
for layer_index, block_output in enumerate(block_outputs, start=1):
    print(f"Output after block {layer_index}, shape [batch, time, channels] = {list(block_output.shape)}:")
    print(block_output[0])


## Step 3 — produce vocabulary logits and measure prediction loss

The final linear layer maps each four-channel representation to five logits, one for each possible next character. Cross-entropy compares each position's logits against its next-character target ID. For example, the first input row `a, b, a, c` is paired with targets `b, a, c, d`. The loss is meaningful as a calculation, but these fixed weights have not learned the language.


In [ ]:
with torch.inference_mode():
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), TARGET_IDS.reshape(-1))

print(f"Next-token target IDs, shape [batch, time] = {list(TARGET_IDS.shape)}:")
print(TARGET_IDS)
print(f"Logits, shape [batch, time, vocab] = {list(logits.shape)} (batch 0):")
print(logits[0])
print(f"Mean next-token cross-entropy on the tiny target batch: {loss.item():.6f}")
print("The output width is the vocabulary size; these fixed weights have not been trained.")


## Step 4 — generate a continuation one token at a time

At each turn, take the logits from the last position, convert them to probabilities, and sample a token. The fixed thresholds make sampling repeatable across languages. Once the context is longer than four tokens, retain only its last four; their position IDs restart at zero in this tiny implementation.


In [ ]:
def sample_from_logits(last_logits, threshold):
    probabilities = F.softmax(last_logits, dim=-1).tolist()
    cumulative_probability = 0.0
    for token_id, probability in enumerate(probabilities):
        cumulative_probability += probability
        if threshold < cumulative_probability:
            return token_id
    return VOCAB_SIZE - 1


def generate(prompt_ids):
    generated = list(prompt_ids)
    with torch.inference_mode():
        for threshold in SAMPLE_THRESHOLDS:
            context = generated[-BLOCK_SIZE:]
            context_ids = torch.tensor([context], dtype=torch.long)
            _, _, next_token_logits = model(context_ids)
            next_token_id = sample_from_logits(next_token_logits[0, -1], threshold)
            generated.append(next_token_id)
    return generated


generated_ids = generate([0, 1])
generated_text = "".join(VOCABULARY[token_id] for token_id in generated_ids)
print("Prompt: ab")
print(f"Generated sequence: {generated_text}")
print("Fixed cumulative-probability thresholds make this untrained demo repeatable across languages.")
